# CityPulse AI — FastAPI Backend
**Run cells top to bottom. Only Cell 5 needs to stay running.**

### Before starting:
1. Upload `citypulse_real_311_scored.csv` to Colab (Files panel → Upload)
2. Run cells 1 → 2 → 3 → 4 → 5 in order
3. Copy the public URL from Cell 5 into your dashboard HTML

## Cell 1 — Install & Clean Up

In [ ]:
# Install packages
!pip install fastapi uvicorn pyngrok pandas -q

# Kill any leftover processes from previous runs
import subprocess
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)

from pyngrok import ngrok
ngrok.kill()

import time
time.sleep(2)

print('✅ Ready')

✅ Ready


## Cell 2 — Upload & Load Your CSV

In [ ]:
import pandas as pd
import requests
import os

CSV_URL = 'https://raw.githubusercontent.com/monkeycruch/CityPulseAI-CityTech-/refs/heads/samples/citypulse_real_311_scored.csv'

if not os.path.exists('citypulse_real_311_scored.csv'):
    print('Downloading CSV from GitHub...')
    r = requests.get(CSV_URL)
    with open('citypulse_real_311_scored.csv', 'wb') as f:
        f.write(r.content)
    print('✅ Downloaded successfully')
else:
    print('✅ CSV already exists locally')

df = pd.read_csv('citypulse_real_311_scored.csv')
cols = ['unique_key','complaint_type','borough','latitude','longitude',
        'priority_score','priority_tier','severity','weather','impact',
        'complaints','accessibility']
df = df[cols].copy()
df = df.rename(columns={
    'unique_key':'id','complaint_type':'complaint',
    'latitude':'lat','longitude':'lon',
    'priority_score':'score','priority_tier':'tier'
})
for col in ['lat','lon','score','severity','weather',
            'impact','complaints','accessibility']:
    df[col] = df[col].round(2)

print(f'✅ Loaded {len(df):,} incidents')
print(df['tier'].value_counts())

✅ Downloaded successfully
✅ Loaded 10,000 incidents
tier
Medium      5293
High        4302
Low          383
Critical      22
Name: count, dtype: int64


## Cell 3 — Write main.py (FastAPI App)

In [ ]:
import os

lines = [
    "from fastapi import FastAPI, Query",
    "from fastapi.middleware.cors import CORSMiddleware",
    "from typing import Optional",
    "import pandas as pd",
    "",
    "df = pd.read_csv('citypulse_real_311_scored.csv')",
    "cols = ['unique_key','complaint_type','borough','latitude','longitude',",
    "        'priority_score','priority_tier','severity','weather','impact',",
    "        'complaints','accessibility']",
    "df = df[cols].copy()",
    "df = df.rename(columns={",
    "    'unique_key':'id','complaint_type':'complaint',",
    "    'latitude':'lat','longitude':'lon',",
    "    'priority_score':'score','priority_tier':'tier'",
    "})",
    "for col in ['lat','lon','score','severity','weather','impact','complaints','accessibility']:",
    "    df[col] = df[col].round(2)",
    "INCIDENTS = df.to_dict(orient='records')",
    "print(f'Loaded {len(INCIDENTS)} incidents')",
    "",
    "app = FastAPI(title='CityPulse AI')",
    "app.add_middleware(CORSMiddleware, allow_origins=['*'],",
    "    allow_credentials=True, allow_methods=['*'], allow_headers=['*'])",
    "",
    "@app.get('/api/health')",
    "async def health():",
    "    return {'status': 'ok', 'total': len(INCIDENTS)}",
    "",
    "@app.get('/api/stats')",
    "async def stats(borough: Optional[str] = None):",
    "    data = [i for i in INCIDENTS if not borough or i['borough']==borough]",
    "    return {'critical': sum(1 for i in data if i['tier']=='Critical'),",
    "            'high': sum(1 for i in data if i['tier']=='High'),",
    "            'medium': sum(1 for i in data if i['tier']=='Medium'),",
    "            'low': sum(1 for i in data if i['tier']=='Low'),",
    "            'total': len(data)}",
    "",
    "@app.get('/api/incidents')",
    "async def get_incidents(tier: Optional[str]=None, borough: Optional[str]=None, limit: int=10000):",
    "    data = INCIDENTS",
    "    if tier: data = [i for i in data if i['tier']==tier]",
    "    if borough: data = [i for i in data if i['borough']==borough]",
    "    return sorted(data, key=lambda x: x['score'], reverse=True)[:limit]",
    "",
    "@app.get('/api/top')",
    "async def top(n: int=20, borough: Optional[str]=None):",
    "    data = [i for i in INCIDENTS if not borough or i['borough']==borough]",
    "    return sorted(data, key=lambda x: x['score'], reverse=True)[:n]",
]

with open('main.py', 'w') as f:
    f.write('\n'.join(lines))

# Verify it loads without errors
import importlib.util
spec = importlib.util.spec_from_file_location('main', 'main.py')
print('✅ main.py written')

# Quick syntax check
import py_compile
try:
    py_compile.compile('main.py', doraise=True)
    print('✅ Syntax check passed')
except py_compile.PyCompileError as e:
    print(f'❌ Syntax error: {e}')

✅ main.py written
✅ Syntax check passed


## Cell 4 — Verify Server Starts Correctly

In [ ]:
import subprocess
import time
import requests

# Start uvicorn
proc = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000', '--log-level', 'error'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for it to start
print('Starting server...')
time.sleep(4)

# Test it
try:
    r = requests.get('http://localhost:8000/api/health', timeout=5)
    print(f'✅ Server is running!')
    print(f'   Response: {r.json()}')
except Exception as e:
    print(f'❌ Server failed to start: {e}')
    # Print stderr for debugging
    err = proc.stderr.read(500).decode()
    if err:
        print(f'   Error: {err}')

Starting server...
✅ Server is running!
   Response: {'status': 'ok', 'total': 10000}


## Cell 5 — Start ngrok & Keep Running
**⚠️ This cell must stay running. Do not stop it.**

In [ ]:
from pyngrok import ngrok
import time

# NOTE: You need to set your ngrok authtoken to use ngrok.connect.
# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
# Replace 'YOUR_AUTHTOKEN' with your actual token.
ngrok.set_auth_token("3DPgO8tivXEFDegSsCNLSlY2YGq_43pHxSiJDFg3BNwX3Fzbf")

# Connect ngrok tunnel
public_url = ngrok.connect(8000, 'http')

# Extract just the URL string
url_str = public_url.public_url if hasattr(public_url, 'public_url') else str(public_url).split('"')[1]

print('\n' + '='*60)
print('✅ CITYPULSE AI BACKEND IS LIVE')
print('='*60)
print(f'\n📍 YOUR PUBLIC URL:')
print(f'\n   {url_str}\n')
print('Paste this URL into your dashboard HTML:')
print(f'   const API_URL = "{url_str}";\n')
print('Test in browser:')
print(f'   {url_str}/api/health')
print(f'   {url_str}/api/stats')
print(f'   {url_str}/docs\n')
print('='*60)
print('⚠️  Keep this cell running!')
print('Server stops when this cell stops.\n')

# Keep alive
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    ngrok.kill()
    print('Server stopped.')


✅ CITYPULSE AI BACKEND IS LIVE

📍 YOUR PUBLIC URL:

   https://macaroni-headcount-crepe.ngrok-free.dev

Paste this URL into your dashboard HTML:
   const API_URL = "https://macaroni-headcount-crepe.ngrok-free.dev";

Test in browser:
   https://macaroni-headcount-crepe.ngrok-free.dev/api/health
   https://macaroni-headcount-crepe.ngrok-free.dev/api/stats
   https://macaroni-headcount-crepe.ngrok-free.dev/docs

⚠️  Keep this cell running!
Server stops when this cell stops.

Server stopped.
